In [7]:
import os
import shutil
from edgar import set_identity, Company
from rich.console import Console
import pandas as pd

set_identity("user@example.com")

DATA_DIR = os.path.join(os.getcwd(), "data", "income_statements")
tickers = [   "WMT"]
console = Console()

In [8]:
for ticker in tickers:
    ticker_dir = os.path.join(DATA_DIR, ticker)
    if os.path.exists(ticker_dir):
        shutil.rmtree(ticker_dir)
    os.makedirs(ticker_dir)

    company = Company(ticker)
    filings = company.get_filings(form="10-K")
    filings_df = filings.to_pandas().head(1)

    for _, row in filings_df.iterrows():
        filing_date = str(row["filing_date"])
        period = str(row["reportDate"])
        accession = row["accession_number"]

        console.rule(f"[bold]{ticker} — Period: {period} (Filed: {filing_date})")

        filing = company.get_filings(form="10-K").get(accession)
        xbrl = filing.xbrl()
        income = xbrl.statements.income_statement(view="detailed")
        console.print(income.render())

        income_df: pd.DataFrame = income.to_dataframe()
        fy_cols = [c for c in income_df.columns if "(FY)" in c]
        income_df = income_df[["label"] + fy_cols]

        # Convert to raw numbers (no formatting) for JSON output
        numeric_df = income_df.copy()
        for col in fy_cols:
            numeric_df[col] = pd.to_numeric(income_df[col], errors="coerce")

        # Add YoY pct change columns (most recent vs prior year)
        if len(fy_cols) >= 2:
            current_col = fy_cols[0]
            prior_col = fy_cols[1]
            yoy_label = f"YoY% ({current_col.split('(')[0].strip()} vs {prior_col.split('(')[0].strip()})"
            numeric_df[yoy_label] = (
                (numeric_df[current_col] - numeric_df[prior_col]) / numeric_df[prior_col].abs()
            ).apply(lambda x: round(x * 100, 2) if pd.notna(x) else None)

            if len(fy_cols) >= 3:
                prior2_col = fy_cols[2]
                yoy2_label = f"YoY% ({prior_col.split('(')[0].strip()} vs {prior2_col.split('(')[0].strip()})"
                numeric_df[yoy2_label] = (
                    (numeric_df[prior_col] - numeric_df[prior2_col]) / numeric_df[prior2_col].abs()
                ).apply(lambda x: round(x * 100, 2) if pd.notna(x) else None)

        display(numeric_df)

        filepath = os.path.join(ticker_dir, f"income_statement_{ticker}.json")
        with open(filepath, "w") as f:
            f.write(numeric_df.to_json(orient="records", indent=2))
        print(f"-> saved {os.path.relpath(filepath)}")

    print()

────────────────────────────────── WMT — Period: 2026-01-31 (Filed: 2026-03-13) ───────────────────────────────────

                                                                                                                   
                                         WALMART INC.   WMT39                                                      
                                         CONSOLIDATED STATEMENT OF INCOME                                          
                                         Jan 31, 2024 to Jan 31, 2026                                              
                                                                                                                   
                                                                     Jan 31, 2026   Jan 31, 2025   Jan 31, 2024    
   ─────────────────────────────────────────────────────────────────────────────────────────────────────────────   
    Revenues:                                                                                                      
        Net sales:                                                       $706,413       $674,538       $642,637    
          Operating Segments - Walmart U.S.                              $482,975       $462,415       $441,817    
          Operating Segments - Walmart International                     $130,423       $121,885       $114,641    
          Operating Segments - Sam's Club U.S.                            $93,015        $90,238        $86,179    
          Grocery - Walmart U.S.                                         $285,482       $276,003       $264,210    
          General merchandise - Walmart U.S.                             $115,060       $113,921       $113,985    
          Health and wellness - Walmart U.S.                              $69,547        $62,092        $54,898    
          Other - Walmart U.S.                                            $12,886        $10,399         $8,724    
          Walmart U.S.                                                   $482,975       $462,415       $441,817    
          E Commerce - Walmart U.S.                                       $99,600        $79,300        $65,400    
          Mexico and Central America - Walmart International              $52,492        $51,970        $49,726    
          China - Walmart International                                   $24,623        $19,975        $17,011    
          Canada - Walmart International                                  $23,724        $23,035        $22,639    
          Other - Walmart International                                   $29,584        $26,905        $25,265    
          Walmart International                                          $130,423       $121,885       $114,641    
          E Commerce - Walmart International                              $35,800        $29,500        $24,800    
          Grocery - Sam's Club U.S.                                       $64,706        $61,253        $57,565    
          Fuel and other - Sam's Club U.S.                                $11,570        $12,960        $13,707    
          General merchandise - Sam's Club U.S.                           $11,549        $11,215        $10,947    
          Health and wellness - Sam's Club U.S.                            $5,190         $4,810         $3,960    
          Sam's Club U.S.                                                 $93,015        $90,238        $86,179    
          E Commerce - Sam's Club U.S.                                    $15,000        $12,100         $9,900    
        Membership and other income:                                       $6,750         $6,447         $5,488    
          Membership Fees                                                  $4,400         $3,800         $3,100    
          Operating Segments - Walmart U.S.                                $2,624         $2,594         $1,985    
          Operating Segments - Walmart International                       $1,565         $1,478         $1,408    
          Operating Segments - Sam's Club U.S.          

,label,2026-01-31 (FY),2025-01-31 (FY),2024-01-31 (FY),YoY% (2026-01-31 vs 2025-01-31),YoY% (2025-01-31 vs 2024-01-31)
0,Revenues:,NaN,NaN,NaN,NaN,NaN
1,Net sales,7.064130e+11,6.745380e+11,6.426370e+11,4.73,4.96
2,Operating Segments - Walmart U.S.,4.829750e+11,4.624150e+11,4.418170e+11,4.45,4.66
3,Operating Segments - Walmart International,1.304230e+11,1.218850e+11,1.146410e+11,7.00,6.32
4,Operating Segments - Sam's Club U.S.,9.301500e+10,9.023800e+10,8.617900e+10,3.08,4.71
...,...,...,...,...,...,...
64,Weighted-average common shares outstanding:,NaN,NaN,NaN,NaN,NaN
65,Basic (in shares),7.983000e+09,8.041000e+09,8.077000e+09,-0.72,-0.45
66,Diluted (in shares),8.022000e+09,8.081000e+09,8.108000e+09,-0.73,-0.33
67,Dividends declared per common share (in USD pe...,9.400000e-01,8.300000e-01,7.600000e-01,13.25,9.21


-> saved data/income_statements/WMT/income_statement_WMT.json

